
# Lab: Explainability & Interpretability for Text Models (SHAP + LIME)

**Duration:** ~1 hour  
**Level:** Intermediate (Python + ML basics)  
**Focus:** Text preprocessing, model training, and local explainability with **LIME** and **SHAP**.



## Objectives
By the end of this lab, you will be able to:
- Distinguish **explainability** vs **interpretability**.
- Preprocess text (TF–IDF) and train a simple classifier.
- Generate **local** text explanations using **LIME** and **SHAP**.
- Interpret which words push a prediction towards a class and compare methods.



## Setup
If needed, install the required packages (uncomment the lines and run):


In [ ]:

# %%capture
# !pip install -U pip
# !pip install scikit-learn lime shap pandas numpy matplotlib



## Background: Key Concepts

| Concept | Description | Example |
|---|---|---|
| **Interpretability** | How easily the internal workings of a model can be understood (often inherent to the model). | Decision Trees expose human-readable rules. |
| **Explainability (XAI)** | Tailors *why* a model made a prediction for a given audience. | SHAP highlights words that most influenced a sentiment label. |
| **Model-Agnostic** | Works with any model without peeking inside. | LIME, SHAP |
| **Local Explanation** | Explains an individual prediction. | "Why is *this* review predicted as negative?" |
| **Global Explanation** | Describes overall model behavior. | "Top global features for sentiment." |

**Explainability vs. Interpretability**  
- **Interpretability**: understanding the internal essence (e.g., linear weights, tree paths).  
- **Explainability**: communicating reasons tailored to a user; can justify or contextualize a specific prediction.  
- They complement each other; XAI is useful to audit black-box models, but some prefer inherently interpretable models to avoid post-hoc pitfalls.



## Data
We'll start with a tiny sentiment dataset for speed. You can later swap in a larger dataset (IMDb, SST, etc.) if time allows.


In [ ]:

import pandas as pd

data = [
    ("I love this movie, it was fantastic!", "positive"),
    ("Absolutely terrible. Waste of time.", "negative"),
    ("The plot was engaging and fun.", "positive"),
    ("I hated the acting.", "negative"),
    ("Wonderful film with great direction.", "positive"),
    ("Not good. Poor script and dull scenes.", "negative"),
    ("Brilliant soundtrack and superb performances.", "positive"),
    ("It was boring and predictable.", "negative"),
    ("A delightful surprise. Highly recommend!", "positive"),
    ("Awful pacing and weak characters.", "negative"),
]

df = pd.DataFrame(data, columns=["text", "label"])
df.head()



## Preprocessing & Model
We'll use a **Pipeline** with `TfidfVectorizer` and `LogisticRegression` for a quick baseline.


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"]
)

pipeline = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    LogisticRegression(max_iter=1000)
)

pipeline.fit(X_train, y_train)
acc = pipeline.score(X_test, y_test)
print(f"Test accuracy: {acc:.3f} (Tiny dataset; don't over-interpret)")



## Local Explanations with LIME
LIME perturbs the input text and learns a simple local surrogate model to explain the prediction.

**Steps:**
1. Pick a test instance.
2. Use `LimeTextExplainer` to explain the model's `predict_proba`.
3. Inspect helpful/harmful words for the predicted class.


In [ ]:

from lime.lime_text import LimeTextExplainer

class_names = ["negative", "positive"]  # order should match model classes
# Ensure class order matches pipeline's classes_
print("Pipeline classes:", list(pipeline.classes_))

explainer = LimeTextExplainer(class_names=class_names)

# Pick an instance to explain
i = 0
text_instance = X_test.iloc[i]
print("Text instance:", text_instance)

exp = explainer.explain_instance(
    text_instance,
    pipeline.predict_proba,
    num_features=8,
    labels=[0, 1]  # indices for classes_
)

# Show explanation inline (works best in a notebook)
# If the HTML view doesn't render in your environment, use exp.as_list(label=...) as a fallback.
display(exp.as_list(label=list(pipeline.classes_).index(pipeline.predict([text_instance])[0])))


In [ ]:

# Optional rich visualization (may open a new window or inline HTML depending on environment)
# exp.show_in_notebook(text=True)



## Local Explanations with SHAP (Text)
SHAP uses game-theoretic Shapley values. We'll use a **text masker** so SHAP can perturb tokens appropriately.

**Notes:**
- We pass `pipeline.predict_proba` so SHAP can handle raw text.
- `shap.plots.text` gives a word-level contribution visualization.


In [ ]:

import shap

# Create a text masker (tokenizer is inferred)
masker = shap.maskers.Text()

# The explainer wraps the model's probability function
explainer = shap.Explainer(pipeline.predict_proba, masker)

# Explain a single instance (list[str])
shap_values = explainer([text_instance])

# Visualize contribution for the first (and only) instance
# If this doesn't render in your environment, try shap.plots.bar or printing shap_values values.
shap.plots.text(shap_values[0])



## Compare LIME vs. SHAP

| Aspect | LIME | SHAP |
|---|---|---|
| Mechanism | Perturbation + local surrogate | Shapley values (game theory) |
| Scope | Local by design | Local & supports global summaries |
| Speed | Usually faster | Can be slower |
| Stability | May vary with sampling | More consistent attribution |
| Output | Word contributions near the instance | Word/token contributions with theoretical grounding |

**Prompts for reflection:**
- Do both methods highlight similar words for your instance?  
- Change the text slightly (e.g., remove a key adjective). Does the explanation flip?  
- Which visualization is clearer for stakeholders (e.g., product managers vs. developers)?



## Try Other Test Instances
Pick several different examples to see how explanations change.


In [ ]:

def explain_text(idx):
    text_i = X_test.iloc[idx]
    print(f"Index: {idx}\nText: {text_i}")
    pred = pipeline.predict([text_i])[0]
    proba = pipeline.predict_proba([text_i])[0]
    print("Prediction:", pred, "Proba:", dict(zip(pipeline.classes_, proba)))

    # LIME quick list view
    exp_i = explainer.explain_instance(text_i, pipeline.predict_proba, num_features=8, labels=[0, 1])
    print("\nLIME (top features for predicted class):")
    print(exp_i.as_list(label=list(pipeline.classes_).index(pred)))

    # SHAP
    shap_vals = explainer([text_i])
    print("\nSHAP values computed. Visualization below (if supported).")
    shap.plots.text(shap_vals[0])

# Try a few
for idx in range(min(3, len(X_test))):
    explain_text(idx)



## (Optional) Global View with SHAP
You can compute SHAP values for many samples and create a global summary. **Note:** can be slow.

Uncomment to try on more data:


In [ ]:

# many_samples = list(X_test)  # or a larger sample from your corpus
# shap_vals_many = explainer(many_samples)
# shap.summary_plot(shap_vals_many)  # bar or beeswarm depending on back-end



## Wrap-Up (5 mins)
- **Explainability vs. Interpretability:** complementary tools for trust and debugging.  
- **Local explanations** help justify individual decisions, especially in sensitive domains.  
- **Next steps:** try SVM/Naive Bayes, bigger datasets, and compare stability across runs.

**Deliverable (if assessed):**
- Submit a short note (≤300 words) comparing LIME vs. SHAP on 2–3 test texts, including screenshots of explanations.
